# 06 — Linear and logistic regression

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/006-linear-and-logistic-regression.ipynb)

Regression is one idea — pick the line with the smallest squared error — applied to two kinds of target. When the target is a number you read the line's height; when the target is yes/no you pass that height through a squashing function and read a probability.

## 1. The numbers: one line through ten points

Ten houses, one feature each, and a price that is roughly three times the size plus five. `np.linalg.lstsq` returns the slope and intercept of the line with the smallest total squared vertical distance to the points, with no loop and no learning rate.

In [ ]:
import numpy as np                                        # the array library
x = np.arange(1, 11, dtype=float)                         # feature: size, in hundreds of square feet
y = 3.0 * x + 5.0 + np.array([-0.2, 0.4, -0.1, 0.3, -0.5, 0.2, 0.1, -0.3, 0.6, -0.4])   # price plus noise
A = np.column_stack([x, np.ones_like(x)])                 # design matrix: a size column, then a column of ones
coef = np.linalg.lstsq(A, y, rcond=None)[0]               # [slope, intercept] with the smallest squared error
print("A shape:", A.shape)                                # ten rows, two columns
print("row 0 of A:", A[0])                                # one row: the size, then the constant 1
print("coefficients:", np.round(coef, 3))                 # slope first, intercept second
print("first three predictions:", np.round(A[:3] @ coef, 3))   # the model is A @ coef
print("last three targets:", np.round(y[-3:], 3))         # the values it approximates

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import numpy as np</code> — loads NumPy.</li><li><code>x = ...</code> — the single feature, sizes 1 to 10.</li><li><code>y = ...</code> — prices from a straight rule, plus small noise.</li><li><code>A = np.column_stack(...)</code> — the two columns the line needs: the feature and a column of ones.</li><li><code>coef = np.linalg.lstsq(...)[0]</code> — least squares solution, slope then intercept.</li><li><code>print("A shape" ...)</code> — ten rows, two unknowns.</li><li><code>print("row 0 of A" ...)</code> — the ones column is how an intercept gets fitted.</li><li><code>print("coefficients" ...)</code> — should land near 3 and 5.</li><li><code>print("first three predictions" ...)</code> — <code>A @ coef</code> applies the line to every row at once.</li><li><code>print("last three targets" ...)</code> — compare predictions with the truth.</li></ul>

</details>

## 2. Step by step: the same answer without lstsq

Least squares has a closed form: build the two-by-two matrix of sums, build the vector of sums, and solve. Accumulating those sums one row at a time shows exactly what `lstsq` is doing, and it lands on the identical slope and intercept.

In [ ]:
AtA = np.zeros((2, 2))                                    # will hold the summed outer products
Aty = np.zeros(2)                                         # will hold the summed feature-times-target
for i in range(len(y)):                                   # one iteration per row
    row = A[i]                                            # this row: [size, 1]
    AtA = AtA + np.outer(row, row)                        # add this row's two-by-two outer product
    Aty = Aty + row * y[i]                                # add this row's contribution to the right side
    print(f"row {i}: AtA[0,0]={AtA[0,0]:.1f}  Aty={np.round(Aty, 1)}")   # watch the sums accumulate
w_hand = np.linalg.solve(AtA, Aty)                        # solve the two equations for slope and intercept
print("by hand:", np.round(w_hand, 3))                    # slope, intercept
print("lstsq  :", np.round(coef, 3))                      # the same numbers, reached differently

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>AtA = np.zeros((2, 2))</code> — the 2-by-2 matrix that will collect the sums.</li><li><code>Aty = np.zeros(2)</code> — the 2-vector on the right-hand side.</li><li><code>for i in range(len(y))</code> — one iteration per row.</li><li><code>row = A[i]</code> — the current row as <code>[size, 1]</code>.</li><li><code>AtA = AtA + np.outer(row, row)</code> — every pair of columns multiplied together.</li><li><code>Aty = Aty + row * y[i]</code> — each column multiplied by this row's target.</li><li><code>print(f"row {i} ...")</code> — the running sums after each row.</li><li><code>w_hand = np.linalg.solve(...)</code> — two equations, two unknowns.</li><li><code>print("by hand" ...)</code> — slope and intercept from the accumulated sums.</li><li><code>print("lstsq" ...)</code> — the same numbers, reached differently.</li></ul>

</details>

## 3. Plot: the logistic S-curve and its boundary

For a yes/no target the line's value is squashed by `1 / (1 + exp(-z))`, which is always between 0 and 1. Fitting the two numbers by gradient descent gives a curve that crosses 0.5 once; that crossing is the decision boundary, and here it sits where the labels switch.

In [ ]:
xc = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0])   # feature: hours studied
yc = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1], dtype=float)              # label: fail or pass

def sigmoid(z):                                           # the squashing function
    return 1 / (1 + np.exp(-z))                           # any real number in, a probability out

wl, bl = 0.0, 0.0                                         # start the logistic weights at zero
for step in range(3000):                                  # gradient descent on the log loss
    p = sigmoid(wl * xc + bl)                             # predicted probability for every row
    wl = wl - 0.5 * np.mean((p - yc) * xc)                # nudge the weight downhill
    bl = bl - 0.5 * np.mean(p - yc)                       # nudge the bias downhill
print("logistic slope, bias:", round(wl, 3), round(bl, 3))   # steepness and offset
print("boundary at x =", round(-bl / wl, 3))              # the input where p equals 0.5
grid = np.linspace(0, 6, 200)                             # a dense grid for drawing
import matplotlib.pyplot as plt                           # the plotting library
plt.figure(figsize=(6, 3))                                # a wide, short figure
plt.plot(grid, sigmoid(wl * grid + bl), color="tab:red", label="predicted probability")   # the S-curve
plt.scatter(xc, yc, color="tab:blue", label="data (0 or 1)")   # the ten labelled rows
plt.axhline(0.5, color="grey", linestyle="--")            # the 50 percent line
plt.axvline(-bl / wl, color="grey", linestyle=":")        # the boundary it defines
plt.xlabel("hours studied")                               # x-axis label
plt.ylabel("probability of passing")                      # y-axis label
plt.legend(fontsize=8)                                    # name the curve and the points
plt.title("One crossing at 0.5 makes the yes/no decision")   # the takeaway
plt.show()                                                # render

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>xc = ...</code> — the feature for the classification problem.</li><li><code>yc = ...</code> — the label, coded 0 for fail and 1 for pass.</li><li><code>def sigmoid(z)</code> — names the squashing function.</li><li><code>return 1 / (1 + np.exp(-z))</code> — the logistic curve, bounded between 0 and 1.</li><li><code>wl, bl = 0.0, 0.0</code> — starting values for the two fitted numbers.</li><li><code>for step in range(3000)</code> — plain gradient descent, no library solver.</li><li><code>p = sigmoid(...)</code> — current probabilities.</li><li><code>wl = wl - ...</code> — the weight moves against the gradient of the log loss.</li><li><code>bl = bl - ...</code> — the bias does the same.</li><li><code>print("logistic slope, bias" ...)</code> — the fitted pair.</li><li><code>print("boundary at x" ...)</code> — solve <code>wl*x + bl = 0</code> for the 50 percent point.</li><li><code>grid = np.linspace(0, 6, 200)</code> — a smooth range for the curve.</li><li><code>import matplotlib.pyplot as plt</code> — the plotting library.</li><li><code>plt.figure(...)</code> — start a figure.</li><li><code>plt.plot(...)</code> — the S-curve across that range.</li><li><code>plt.scatter(...)</code> — the data, sitting at exactly 0 or 1.</li><li><code>plt.axhline(0.5, ...)</code> — the 50 percent decision line.</li><li><code>plt.axvline(...)</code> — the boundary it implies.</li><li><code>plt.xlabel(...)</code> — the input.</li><li><code>plt.ylabel(...)</code> — the fitted probability.</li><li><code>plt.legend(...)</code> — names the curve and the points.</li><li><code>plt.title(...)</code> — states what the crossing means.</li><li><code>plt.show()</code> — renders the figure.</li></ul>

</details>

## 4. Interactive: move the threshold

0.5 is a convention, not a law. The slider changes the probability you require before saying yes; the green line slides along the curve and the points recolour as their predictions flip. The title reports the accuracy you get at that threshold.

In [ ]:
from ipywidgets import interact                            # sliders that rerun a function
grid = np.linspace(0, 6, 200)                             # the same dense drawing grid
p_all = sigmoid(wl * xc + bl)                             # reuse the fit from section 3

def show(thr=0.5):                                        # thr comes from the slider
    pred = (p_all >= thr).astype(int)                     # 1 where the probability clears the bar
    acc = np.mean(pred == yc)                             # share of rows called correctly
    cut = (np.log(thr / (1 - thr)) - bl) / wl             # invert the sigmoid to find the x value
    plt.figure(figsize=(6, 3))                            # a fresh figure for each drag
    plt.plot(grid, sigmoid(wl * grid + bl), color="tab:red")   # the fitted S-curve
    plt.axhline(thr, color="tab:orange", linestyle="--")  # your chosen probability
    plt.axvline(cut, color="tab:green", linestyle=":")    # where the curve crosses it
    plt.scatter(xc, yc, c=pred, cmap="coolwarm", edgecolor="k", s=60)   # colour = prediction
    plt.xlabel("hours studied")                           # x-axis label
    plt.ylabel("probability of passing")                  # y-axis label
    plt.title(f"threshold {thr:.2f}: boundary at {cut:.2f}, accuracy {acc:.0%}")   # live readout
    plt.show()                                            # draw the frame

interact(show, thr=(0.1, 0.9, 0.05))                      # attach the slider

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>from ipywidgets import interact</code> — the widget helper.</li><li><code>grid = np.linspace(0, 6, 200)</code> — the drawing grid.</li><li><code>p_all = sigmoid(...)</code> — the ten fitted probabilities, computed once.</li><li><code>def show(thr=0.5)</code> — redraws one frame per slider value.</li><li><code>pred = (...).astype(int)</code> — hard 0/1 decisions at this threshold.</li><li><code>acc = np.mean(...)</code> — accuracy, as a fraction of ten rows.</li><li><code>cut = ...</code> — inverts the sigmoid: <code>log(thr/(1-thr))</code> is the <code>z</code> needed, then solve for <code>x</code>.</li><li><code>plt.figure(...)</code> — a fresh figure per drag.</li><li><code>plt.plot(...)</code> — the unchanging curve.</li><li><code>plt.axhline(thr, ...)</code> — your threshold as a horizontal line.</li><li><code>plt.axvline(cut, ...)</code> — the boundary it produces.</li><li><code>plt.scatter(..., c=pred, cmap=...)</code> — points coloured by their decision.</li><li><code>plt.xlabel(...)</code> — the input.</li><li><code>plt.ylabel(...)</code> — the fitted probability.</li><li><code>plt.title(f"...")</code> — threshold, boundary and accuracy, updated live.</li><li><code>plt.show()</code> — draws the frame.</li><li><code>interact(show, thr=(...))</code> — slider from 0.1 to 0.9 in steps of 0.05.</li></ul>

</details>

## 5. When it breaks: squared error on a 0/1 target

Squared error treats a probability like any other number, so it is happy with predictions above 1 and below 0 — which is what least squares produces on labels. Worse, it charges a confidently wrong probability less than one unit, while log loss grows without limit.

In [ ]:
Ac = np.column_stack([xc, np.ones_like(xc)])              # design matrix for the 0/1 data
line = np.linalg.lstsq(Ac, yc, rcond=None)[0]             # best straight line for the labels
fit = Ac @ line                                           # its outputs are heights, not probabilities
print("fitted line:", np.round(line, 3))                  # slope and intercept
print("smallest prediction:", round(fit.min(), 3))        # below zero: impossible
print("largest  prediction:", round(fit.max(), 3))        # above one: impossible
print("MSE  for p=0.40 when y=1:", round((0.40 - 1) ** 2, 3))    # a mild miss
print("MSE  for p=0.02 when y=1:", round((0.02 - 1) ** 2, 3))    # a confident miss, barely worse
print("log loss for p=0.40:", round(-np.log(0.40), 3))    # mild miss
print("log loss for p=0.02:", round(-np.log(0.02), 3))    # punished far harder
plt.figure(figsize=(6, 3))                                # a wide, short figure
plt.scatter(xc, yc, color="tab:blue", label="labels")     # the 0/1 truth
plt.plot(xc, fit, color="tab:red", label="least squares line")   # the line that ignores the bounds
plt.axhspan(0, 1, color="tab:green", alpha=0.12, label="valid probability band")   # 0 to 1
plt.xlabel("hours studied")                               # x-axis label
plt.ylabel("prediction")                                  # y-axis label
plt.legend(fontsize=8, loc="upper left")                  # name the elements
plt.title("A line on 0/1 labels predicts probabilities below 0 and above 1")   # the takeaway
plt.show()                                                # render

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>Ac = np.column_stack(...)</code> — same design matrix shape as section 1.</li><li><code>line = np.linalg.lstsq(...)</code> — least squares applied to yes/no labels.</li><li><code>fit = Ac @ line</code> — the fitted values themselves.</li><li><code>print("fitted line" ...)</code> — slope and intercept of that line.</li><li><code>print("smallest prediction" ...)</code> — the lowest fitted value, negative.</li><li><code>print("largest prediction" ...)</code> — the highest, above 1.</li><li><code>print("MSE for p=0.40" ...)</code> — squared error on a mild miss.</li><li><code>print("MSE for p=0.02" ...)</code> — squared error on a confident miss: still under 1.</li><li><code>print("log loss for p=0.40" ...)</code> — the same mild miss under log loss.</li><li><code>print("log loss for p=0.02" ...)</code> — the confident miss, punished far harder.</li><li><code>plt.figure(...)</code> — a wide, short figure.</li><li><code>plt.scatter(...)</code> — the labels at 0 and 1.</li><li><code>plt.plot(xc, fit, ...)</code> — the straight line, visibly leaving the legal band.</li><li><code>plt.axhspan(0, 1, ...)</code> — shades the only values a probability may take.</li><li><code>plt.xlabel(...)</code> — the input.</li><li><code>plt.ylabel(...)</code> — the fitted height, not a probability.</li><li><code>plt.legend(...)</code> — names the points, the line and the band.</li><li><code>plt.title(...)</code> — names the failure.</li><li><code>plt.show()</code> — renders the figure.</li></ul>

</details>

## Your turn

Set `yc[5]` to 0 in section 3 and refit: watch the boundary move and the accuracy in section 4 peak at a threshold other than 0.5. Then compute the log loss of the least squares line in section 5 after clipping its predictions to `[0.001, 0.999]` — why is that clipping a patch, not a fix?